# Part 5d: FIFO depth optimization

With `io_stream`, every connection between two layers is a FIFO. hls4ml sizes these FIFOs conservatively, which wastes BRAM. The `vitisunified:fifo_depth_optimization` flow runs an RTL co-simulation with FIFO profiling enabled, records the maximum occupancy of every FIFO, and writes the optimized depths to `fifo_depths.json` in the project directory. These depths are then applied to the design.

This notebook enables the flow and checks that the file is produced. Because the flow includes a co-simulation, it takes about as long as Part 5c.

## Settings

Set the target board here. The backend picks the platform (`.xpfm` or `.xsa`) from its `supported_boards.json` using the board and the AXI mode, so nothing else needs to be configured. To add another board, see `5_vu_f_platform_setup.md`.

| Variable | Meaning |
|----------|---------|
| `BOARD` | `'zcu102'` or `'kv260'` |
| `AXI_MODE` | `'axi_master'` (default) or `'axi_stream'`; see the *AXI interface modes* page of the hls4ml docs |

The Vitis tools are located through the `XILINX_VITIS` environment variable, as in the other parts of this tutorial.

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import sys
import json

sys.path.append('../..')
from models import create_simple_unet
import hls4ml

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

BOARD = 'zcu102'  # or 'kv260'
AXI_MODE = 'axi_master'  # or 'axi_stream'

DATA_DIR = '../../data/vitis_unified'
os.makedirs(DATA_DIR, exist_ok=True)
np.random.seed(0)

## Create the model and some test data

We use a tiny U-Net with a single skip connection, defined in `models.py`. The weights are left untrained on purpose: the goal of this part is to check that every stage of the accelerator flow reproduces the same numbers, not to reach a good accuracy. A `Concatenate` skip connection is a good stress test for the streaming interface, because it forces two branches to be merged again.

In [ ]:
X = np.random.rand(10, 4, 4, 1).astype(np.float32)
np.save(f'{DATA_DIR}/X_part5d.npy', X)
print(f'Input shape: {X.shape}')

model = create_simple_unet()
model.summary()

## Reference predictions for the testbench

The testbench generated by hls4ml reads its inputs and expected outputs from files. We pass them with `input_data_tb` and `output_data_tb`. The expected outputs are produced with the C++ bridge, exactly as in Part 5a, so the comparison checks the tool flow rather than the model.

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')

# Reference predictions from the C++ bridge
hls_model_ref = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='VitisUnified',
    io_type='io_stream',
    output_dir=os.path.abspath(f'../../hls4ml_prjs/hls4ml_prj_part5d_ref_{AXI_MODE}'),
    board=BOARD,
    axi_mode=AXI_MODE,
    input_type='float',
    output_type='float',
    clock_period=10,
)
hls_model_ref.compile()
y_ref = hls_model_ref.predict(X)
np.save(f'{DATA_DIR}/y_part5d.npy', y_ref)
print(f'Reference predictions saved: shape {np.array(y_ref).shape}')

## Enable the FIFO depth optimization flow

The optimization is an hls4ml *flow*, enabled through the `Flows` key of the configuration. It runs when the model is compiled, so the co-simulation happens inside `compile()` here. **This can take 10-20 minutes.**

In [ ]:
config['Flows'] = ['vitisunified:fifo_depth_optimization']

In [ ]:
output_dir = os.path.abspath(f'../../hls4ml_prjs/hls4ml_prj_part5d_{AXI_MODE}')

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='VitisUnified',
    io_type='io_stream',
    output_dir=output_dir,
    board=BOARD,
    axi_mode=AXI_MODE,
    input_type='float',
    output_type='float',
    input_data_tb=f'{DATA_DIR}/X_part5d.npy',
    output_data_tb=f'{DATA_DIR}/y_part5d.npy',
    clock_period=10,
)
hls_model.compile()

## Check the FIFO depths file

In [ ]:
fifo_depths_path = f'{output_dir}/fifo_depths.json'
assert os.path.exists(fifo_depths_path), f'fifo_depths.json not found at {fifo_depths_path}'

with open(fifo_depths_path) as f:
    fifo_depths = json.load(f)

print('FIFO depths:')
for name, depth in fifo_depths.items():
    print(f'  {name}: {depth}')